# Ćwiczenie 2 — Sentinel-2: klasyfikacja nienadzorowana K-means

**Cel:** przygotować mapę klastrów spektralnych na podstawie Sentinel-2 SR po maskowaniu chmur, oszacować powierzchnie klastrów i zapisać wynik do GeoTIFF. Klasyfikacja wykonuje się **lokalnie** (`scikit-learn`), dane pochodzą z **GEE**.

**Uwaga:** klastry K-means **nie są automatycznie klasami LULC** — ich interpretacja wymaga porównania z RGB i innymi danymi.

## 1. Importy i ustawienia

Biblioteki instalujemy wcześniej z `requirements.txt`, nie w notebooku. Usuwamy ewentualne konflikty danych PROJ/GDAL po PostgreSQL.

In [1]:
import os
for key in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    os.environ.pop(key, None)

from pathlib import Path
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

PROJECT_ID = "ee-beatastark"
OUTPUT_DIR = Path("results_kmeans")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Przygotowano środowisko.")

Przygotowano środowisko.


## 2. Połączenie z GEE

Jeśli uwierzytelnienie wygasło, odkomentuj `ee.Authenticate(auth_mode="localhost")` i uruchom komórkę.

In [2]:
# ee.Authenticate(auth_mode="localhost")  # tylko jeśli potrzebne
ee.Initialize(project=PROJECT_ID)
print("GEE:", ee.Number(2).add(3).getInfo())

GEE: 5


## 3. Sentinel-2 i maskowanie SCL

Wybieramy tę samą lokalizację co w poprzednim ćwiczeniu. Do klasyfikacji używamy sześciu kanałów; SCL ma rozdzielczość 20 m, dlatego podczas eksportu zostanie dopasowany do siatki 10 m.

In [3]:
center = ee.Geometry.Point([19.94, 50.06])
roi = center.buffer(3000).bounds()  # 6 × 6 km (w przybliżeniu)
bands = ["B2", "B3", "B4", "B8", "B11", "B12"]

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(roi)
      .filterDate("2026-08-01", "2026-09-01")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)))

count = s2.size().getInfo()
print("Liczba scen:", count)
if count == 0:
    raise ValueError("Brak scen: zmień zakres dat lub limit zachmurzenia.")

image = ee.Image(s2.sort("CLOUDY_PIXEL_PERCENTAGE").first())
print("Data:", ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo())

scl = image.select("SCL")
valid_scl = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)))
# Wartości SCL: 3 cień, 8/9 chmury, 10 cirrus.
# Śnieg (11) pozostawiamy; można dodać .And(scl.neq(11)).

reflectance = image.select(bands).multiply(0.0001).updateMask(valid_scl)
print("Gotowe do pobrania (GEE wykonuje operacje na żądanie).")

Liczba scen: 5
Data: 2026-08-14
Gotowe do pobrania (GEE wykonuje operacje na żądanie).


## 4. Podgląd RGB w GEE

In [5]:
import geemap
import geemap.basemaps as basemaps

print("geemap:", geemap.__version__)
print("Plik:", geemap.__file__)
print("basemaps:", type(basemaps))
print("xyz_to_folium:", hasattr(basemaps, "xyz_to_folium"))

geemap: 0.38.9
Plik: C:\Users\python_course\python_gis\envs\jupyter_base\Lib\site-packages\geemap\__init__.py
basemaps: <class 'box.box.Box'>
xyz_to_folium: False


In [6]:
import importlib
import geemap

basemaps_module = importlib.import_module("geemap.basemaps")
geemap.basemaps = basemaps_module

import geemap.foliumap as geemap_folium

In [7]:
import geemap.foliumap as geemap_folium

m = geemap_folium.Map(center=[50.06, 19.94], zoom=11)

m.addLayer(
    image.clip(roi),
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "RGB — oryginał"
)

m.addLayer(
    image.updateMask(valid_scl).clip(roi),
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "RGB — po masce SCL",
    False
)

m.addLayerControl()
m

## 5. Eksport sześciu kanałów do GeoTIFF

Pobieramy fragment w układzie **UTM 34N (EPSG:32634)**, z rozdzielczością 10 m. Brak danych zapisujemy jako `-9999`, co upraszcza późniejsze maskowanie. Operacja może potrwać zależnie od rozmiaru wycinka.

In [ ]:
stack_path = OUTPUT_DIR / "sentinel2_6bands_clear.tif"

export_image = reflectance.unmask(-9999).toFloat().clip(roi)
geemap.download_ee_image(
    export_image,
    filename=str(stack_path),
    region=roi,
    crs="EPSG:32634",
    scale=10,
    dtype="float32",
    overwrite=True,
)
print("Zapisano:", stack_path.resolve())

## 6. Wczytanie rastra i wybranie poprawnych pikseli

`rasterio.read()` zwraca tablicę `(kanał, wiersz, kolumna)`. K-means wymaga macierzy `(piksel, cecha)`.

In [ ]:
with rasterio.open(stack_path) as src:
    data = src.read().astype("float32")
    profile = src.profile.copy()
    transform = src.transform
    crs = src.crs
    print("Kanały, wiersze, kolumny:", data.shape)
    print("CRS:", crs, "| rozdzielczość:", src.res)

valid = np.all(np.isfinite(data) & (data != -9999), axis=0)
rows, cols = valid.shape
X = np.moveaxis(data, 0, -1)[valid]  # (n_ważnych_pikseli, 6)
print("Liczba poprawnych pikseli:", X.shape[0])
if X.shape[0] < 100:
    raise ValueError("Za mało poprawnych pikseli — wybierz inną scenę/obszar.")

## 7. K-means: standaryzacja i klasyfikacja

Standaryzujemy kanały, aby różne zakresy wartości nie wpływały nieproporcjonalnie na odległość euklidesową. Model uczymy na losowej próbce, a etykiety obliczamy dla wszystkich poprawnych pikseli.

In [ ]:
N_CLUSTERS = 6
MAX_TRAIN = 30_000
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)
train_idx = rng.choice(len(X), size=min(len(X), MAX_TRAIN), replace=False)
scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx])

model = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
model.fit(X_train)

labels = model.predict(scaler.transform(X)).astype("uint8") + 1  # 1..K
lulc_clusters = np.zeros((rows, cols), dtype="uint8")  # 0 = NoData
lulc_clusters[valid] = labels
print("Klastry:", np.unique(lulc_clusters, return_counts=True))

## 8. Mapa klastrów i porównanie z RGB

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

rgb = np.moveaxis(data[[2, 1, 0]], 0, -1)  # B4, B3, B2
rgb = np.clip(rgb / 0.30, 0, 1)
rgb[~valid] = 0

palette = plt.get_cmap("tab10", N_CLUSTERS)
cluster_masked = np.ma.masked_equal(lulc_clusters, 0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(rgb)
axes[0].set_title("Sentinel-2 RGB (orientacyjnie)")
im = axes[1].imshow(cluster_masked, cmap=palette, vmin=0.5, vmax=N_CLUSTERS+0.5,
                    interpolation="nearest")
axes[1].set_title(f"K-means: {N_CLUSTERS} klastrów")
fig.colorbar(im, ax=axes[1], ticks=range(1, N_CLUSTERS+1), label="Klaster")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Powierzchnia klastrów

Dla rastra w metrycznym CRS liczymy powierzchnię z wymiarów piksela. **Nie** wliczamy klasy 0 (NoData).

In [ ]:
pixel_area_m2 = abs(transform.a * transform.e - transform.b * transform.d)
print(f"Powierzchnia piksela: {pixel_area_m2:.2f} m²")

for cluster in range(1, N_CLUSTERS + 1):
    number = int(np.count_nonzero(lulc_clusters == cluster))
    print(f"Klaster {cluster}: {number:7d} pikseli, {number * pixel_area_m2 / 1e6:.3f} km²")

## 10. Zapis mapy klastrów do GeoTIFF

Raster wyjściowy ma identyczny CRS, transformację i rozmiar jak dane wejściowe. `0` oznacza NoData.

In [ ]:
out_path = OUTPUT_DIR / "sentinel2_kmeans_clusters.tif"
out_profile = profile.copy()
out_profile.update(count=1, dtype="uint8", nodata=0, compress="deflate", predictor=1)
with rasterio.open(out_path, "w", **out_profile) as dst:
    dst.write(lulc_clusters, 1)
print("Zapisano:", out_path.resolve())

with rasterio.open(out_path) as src:
    print("Kontrola:", src.count, src.crs, src.res, src.nodata)

## 11. Interpretacja klastrów — zadania

- [ ] Zmień `N_CLUSTERS` z 6 na 5 i 7, porównaj mapy i powierzchnie.
- [ ] Zlokalizuj klastry odpowiadające prawdopodobnie roślinności, zabudowie i wodzie, porównując z RGB. **Nie przypisuj nazw automatycznie.**
- [ ] Sprawdź, jakie wartości reflektancji mają centra klastrów (`scaler.inverse_transform(model.cluster_centers_)`).
- [ ] Zastanów się, czy pojedyncza scena wystarcza do uzyskania wiarygodnej mapy LULC.
- [ ] Wyjaśnij, dlaczego wszystkie kanały muszą mieć wspólną siatkę i dlaczego NoData nie może uczestniczyć w klasyfikacji.

**Do kolejnego ćwiczenia MCDA:** `results_kmeans/sentinel2_kmeans_clusters.tif` będzie jednym z rastrów wejściowych, ale najpierw trzeba zweryfikować i zinterpretować klastry jako kategorie pokrycia terenu.